# Vision-Language Models: The Code Companion

*Runnable, heavily-commented, from-first-principles code for every idea in the [Theory Companion](./VLM_Theory_Companion.md), paired with production-style annotated pipelines for CLIP and Qwen2.5-VL.*

**Structure of this notebook:**

- **Part A — Build it yourself (fully runnable here, NumPy only, no GPU/internet needed).** You will implement cosine similarity, self-attention, and CLIP's contrastive loss from scratch, in ~15 lines each, and verify they behave the way the theory predicts.
- **Part B — CLIP zero-shot classification, annotated end-to-end.** The real `transformers` pipeline, with every line explained and tied back to Part A.
- **Part C — Qwen2.5-VL image captioning, annotated end-to-end.**
- **Part D — Qwen2.5-VL object detection, annotated end-to-end**, including a robust JSON parser explained line by line.

Parts B–D require downloading real model weights and a GPU (or patience on CPU) — run them in your bootcamp environment (Kaggle/Colab). Part A runs anywhere, including right here, with nothing but NumPy, so you can build intuition before spending any compute.


---
## Part A — Build It Yourself

The goal here isn't to reimplement CLIP — it's to implement the three ideas that make CLIP (and every embedding-based system) work, in isolation, so you can watch them behave correctly on toy data before trusting them inside a 400-million-parameter model.


### A1. Cosine similarity, from scratch

**Theory recap:** cosine similarity measures the *angle* between two vectors, ignoring their length. We normalize each vector to unit length (dividing by its own norm), then take the dot product — the dot product of two unit vectors *is* the cosine of the angle between them.

Watch the code do exactly what the formula says, line by line:
- `np.linalg.norm(a, axis=-1, keepdims=True)` computes each vector's length (its "magnitude").
- Dividing by that length rescales every vector to length 1, without changing its direction.
- `a_norm @ b_norm.T` is a matrix of dot products between every pair — since both sides are already unit length, this dot product *is* the cosine similarity, no further division needed.


In [ ]:
import numpy as np

def cosine_similarity(a, b):
    """
    a: shape (n, d) - n vectors of dimension d  (e.g. n text embeddings)
    b: shape (m, d) - m vectors of dimension d  (e.g. m image embeddings)
    returns: shape (n, m) similarity matrix, values in [-1, 1]
    """
    a_norm = a / np.linalg.norm(a, axis=-1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=-1, keepdims=True)
    return a_norm @ b_norm.T

# --- sanity checks that prove the implementation matches the theory ---
rng = np.random.default_rng(0)
text_vecs = rng.normal(size=(4, 8))
image_vecs = rng.normal(size=(4, 8))

sim = cosine_similarity(text_vecs, image_vecs)
print("Similarity matrix shape:", sim.shape)   # -> (4, 4): every text vs every image
print(sim.round(3))

# a vector compared against itself must have similarity exactly 1.0 (angle = 0)
self_sim = cosine_similarity(text_vecs, text_vecs)
assert np.allclose(np.diag(self_sim), 1.0)
print("\nSelf-similarity check passed: every vector has cosine similarity 1.0 with itself.")


**What to notice in the output:** the diagonal check passing is the whole point of Part 4.3 in the theory doc made concrete — identical direction always scores exactly 1.0, regardless of the vectors' actual magnitude (try scaling `text_vecs` by 100 and rerunning; the similarity matrix won't change at all).

### A2. Self-attention, from scratch

**Theory recap:** every token produces a Query, a Key, and a Value. A token's new representation is a weighted sum of *every* token's Value, where the weight comes from how well that token's Query matches each other token's Key.

Implementation walkthrough:
- `Q = X @ Wq`, `K = X @ Wk`, `V = X @ Wv` — three different linear projections of the same input, giving each token three different "views" of itself.
- `Q @ K.T` — a full `(seq_len, seq_len)` matrix of every Query-Key dot product: "how relevant is token *j* to token *i*?"
- Dividing by `sqrt(d_k)` (**scaled** dot-product attention) keeps the dot products from growing too large as dimensionality increases, which would otherwise push the softmax into a near one-hot regime and stall learning.
- `softmax(scores, axis=-1)` turns each row into a proper probability distribution over "which tokens should I attend to."
- `weights @ V` — the actual attention step: each token's output is a weighted blend of every token's Value vector.


In [ ]:
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)  # subtract max for numerical stability
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def self_attention(X, Wq, Wk, Wv):
    """
    X:            (seq_len, d_model) - input token embeddings (text tokens OR image patches)
    Wq, Wk, Wv:   (d_model, d_k)     - learned projection matrices
    returns:      (seq_len, d_k) updated representations, and the attention weight matrix
    """
    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv
    d_k = Wk.shape[1]
    scores = (Q @ K.T) / np.sqrt(d_k)     # scaled dot-product similarity between every pair of tokens
    weights = softmax(scores, axis=-1)    # each row: a probability distribution over "what to attend to"
    output = weights @ V                  # blend Values according to those probabilities
    return output, weights

seq_len, d_model, d_k = 5, 16, 8
X = rng.normal(size=(seq_len, d_model))                 # 5 tokens, 16-dim embeddings (toy sequence)
Wq = rng.normal(size=(d_model, d_k)) * 0.1
Wk = rng.normal(size=(d_model, d_k)) * 0.1
Wv = rng.normal(size=(d_model, d_k)) * 0.1

out, attn_weights = self_attention(X, Wq, Wk, Wv)
print("Output shape:", out.shape, "  <- one updated vector per input token")
print("\nAttention weight matrix (row i = how token i distributes attention across all tokens):")
print(attn_weights.round(2))
print("\nEach row sums to 1 (it's a probability distribution):", attn_weights.sum(axis=1).round(3))


**What to notice:** row *i* of `attn_weights` tells you exactly how token *i*'s new representation was built — as a weighted mixture of all 5 tokens' Value vectors. This is the *same* mechanism whether `X` holds word tokens (CLIP's text encoder, Qwen's language model) or image patch tokens (CLIP's / Qwen's vision encoder) — self-attention doesn't know or care what modality it's mixing, which is exactly why Transformers generalize so well across text and vision.

### A3. CLIP's contrastive (InfoNCE-style) loss, from scratch

**Theory recap:** for a batch of *N* (image, text) pairs, CLIP builds an *N × N* similarity matrix, treats the diagonal as the correct matches, and runs cross-entropy in *both directions* (image→text and text→image), averaging the two.

Implementation walkthrough:
- Normalize both embedding sets (unit vectors, same reasoning as A1).
- `logits = (image_embeds @ text_embeds.T) / temperature` — the full similarity matrix, scaled by a temperature. A **smaller** temperature makes the softmax sharper (more confident, more punishing of near-misses); CLIP learns this value during training rather than fixing it by hand.
- `labels = np.arange(N)` — the correct answer for row *i* is column *i*, by construction (pair *i* is matched with pair *i*).
- Cross-entropy in one direction treats each **image** as a query trying to find its caption among `N` candidates; the transposed version does the same treating each **caption** as a query trying to find its image. Averaging both directions is what makes the loss *symmetric*.


In [ ]:
def contrastive_loss(image_embeds, text_embeds, temperature=0.07):
    """
    image_embeds, text_embeds: (N, d) - matched pairs share the same row index
    Implements CLIP's symmetric InfoNCE-style loss.
    """
    image_embeds = image_embeds / np.linalg.norm(image_embeds, axis=-1, keepdims=True)
    text_embeds = text_embeds / np.linalg.norm(text_embeds, axis=-1, keepdims=True)

    logits = (image_embeds @ text_embeds.T) / temperature   # (N, N) scaled similarity matrix
    N = logits.shape[0]
    labels = np.arange(N)                                    # correct match for row i is column i

    def cross_entropy(logits, labels):
        probs = softmax(logits, axis=-1)
        correct_probs = probs[np.arange(len(labels)), labels]
        return -np.mean(np.log(correct_probs + 1e-9))

    loss_i2t = cross_entropy(logits, labels)      # "does each image find its caption?"
    loss_t2i = cross_entropy(logits.T, labels)    # "does each caption find its image?"
    return (loss_i2t + loss_t2i) / 2

rng = np.random.default_rng(1)

# Case 1: embeddings that are ALREADY well-aligned (matched pairs point the same way)
matched_img = rng.normal(size=(4, 64))
matched_txt = matched_img + rng.normal(size=(4, 64)) * 0.05   # nearly identical direction
loss_aligned = contrastive_loss(matched_img, matched_txt)

# Case 2: completely unrelated embeddings (what an untrained model looks like)
random_img = rng.normal(size=(4, 64))
random_txt = rng.normal(size=(4, 64))
loss_random = contrastive_loss(random_img, random_txt)

print(f"Loss with well-aligned pairs:   {loss_aligned:.4f}   <- near zero: model is 'confident and correct'")
print(f"Loss with unaligned pairs:      {loss_random:.4f}   <- much higher: this is the signal that drives learning")
print(f"(for reference, ln(N) = ln(4) = {np.log(4):.4f} is the loss of pure random guessing)")


**What to notice:** this is the *entire* training signal that shaped CLIP's 400M-example run. Nothing here is specific to images or text — swap `matched_img`/`matched_txt` for any two sets of paired vectors and the same loss function would train any dual-encoder contrastive system (this exact pattern is reused for audio-text, video-text, and code-text models). What makes CLIP *CLIP* is purely the choice of encoders (ViT + text Transformer) feeding into this loss, not the loss itself.

---
## Part B — CLIP Zero-Shot Classification, Annotated End-to-End

> **Note:** this section downloads real model weights and needs internet access + a `transformers` install — run it in your Kaggle/Colab bootcamp environment, not required to run here. Every line is commented to explain *what* it does and *why*, tying back to Parts A1 and A2.


In [ ]:
# pip install transformers pillow matplotlib seaborn torch  (run once, in your bootcamp environment)

from transformers import CLIPTokenizer, CLIPProcessor, CLIPModel
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

# ---- 1. Load the model ----
# "clip-vit-base-patch32": a Vision Transformer that slices images into 32x32 patches (Part 2.2 of the
# theory doc), paired with a text Transformer. Both output 512-dim vectors into a SHARED space (Part 4.1).
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name)
model.eval()  # disables dropout / other training-only behavior -- always do this for inference

tokenizer = CLIPTokenizer.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)  # handles IMAGE preprocessing (resize/normalize/patch)


In [ ]:
# ---- 2. Candidate labels, wrapped in a template ----
# Per Part 4.4 / Part 6 of the theory doc: bare words like "bird" underperform templated phrases like
# "a photo of a bird", because CLIP's training captions were natural sentences, not single words.
# This IS prompt engineering -- we are steering CLIP toward the phrasing distribution it was trained on.
class_names = ["bird", "crow", "car", "bus"]
text_prompts = [f"a photo of a {name}" for name in class_names]

text_inputs = tokenizer(text_prompts, padding=True, return_tensors="pt")
# padding=True: pads shorter prompts to the same length as the longest one in the batch (Part 2.1),
# since the model processes the whole batch as one fixed-size tensor.

with torch.no_grad():          # inference only -- don't build a computation graph for backprop, saves memory
    text_embeddings = model.get_text_features(**text_inputs)     # -> shape (4, 512)

print("Text embeddings shape:", text_embeddings.shape)


In [ ]:
# ---- 3. Encode the image the same way ----
image = Image.open("your_image.jpg").convert("RGB")   # replace with your actual image path

image_inputs = processor(images=image, return_tensors="pt")
# processor does: resize to the model's expected size, normalize pixel values, and slice into
# patches -- the exact patching step described in Part 2.2 of the theory doc.

with torch.no_grad():
    image_embeddings = model.get_image_features(**image_inputs)  # -> shape (1, 512)

print("Image embedding shape:", image_embeddings.shape)


In [ ]:
# ---- 4. Classify: this is Part A1's cosine_similarity(), just now on real 512-dim CLIP vectors ----
similarity = F.cosine_similarity(
    image_embeddings,                 # (1, 512)
    text_embeddings,                  # (4, 512)
    dim=-1
)
# NOTE: torch's F.cosine_similarity here broadcasts one image against all 4 text vectors in one call --
# mathematically identical to the cosine_similarity() function you wrote by hand in Part A1.

probs = similarity.softmax(dim=-1)    # turn raw similarity scores into a probability distribution
predicted_idx = probs.argmax().item()

print("Similarity scores:", similarity.detach().numpy().round(3))
print("Probabilities:    ", probs.detach().numpy().round(3))
print(f"\nPredicted class: '{class_names[predicted_idx]}'  (confidence: {probs[predicted_idx]:.1%})")

# This is the ENTIRE "classifier" -- there is no trained classification head. Swap class_names for any
# new list of words and you have a brand-new zero-shot classifier with zero additional training (Part 4.4).


### B.1 Visualizing many-vs-many similarity (the heatmap pattern)

The single-image classification above is really just one row of a larger *N images × M texts* similarity matrix. Computing the full matrix at once — and visualizing it — makes CLIP's learned geometry directly inspectable, which is a genuinely useful debugging habit: if semantically related classes (e.g. "bird"/"crow") don't show elevated similarity to each other's *images*, something about your inputs or prompts is off.


In [ ]:
# ---- Full N x M similarity heatmap across several images and several text prompts ----
image_paths = {
    "bird": "Bird_image.jpg",
    "crow": "Crow_image.jpg",
    "car":  "Car_image.jpeg",
    "bus":  "Bus_image.jpg",
}
images = [Image.open(p).convert("RGB") for p in image_paths.values()]
labels = list(image_paths.keys())

image_inputs = processor(images=images, return_tensors="pt")
with torch.no_grad():
    all_image_embeddings = model.get_image_features(**image_inputs)     # (4, 512)

# text_embeddings from the earlier cell is (4, 512) already, in the same class order
sim_matrix = F.cosine_similarity(
    text_embeddings[:, None, :],      # reshape to (4, 1, 512) ...
    all_image_embeddings[None, :, :], # ... vs (1, 4, 512) -> broadcasts to a full (4, 4) matrix
    dim=2
).detach().numpy()

plt.figure(figsize=(6, 5))
sns.heatmap(sim_matrix, annot=True, fmt=".2f",
            xticklabels=labels, yticklabels=class_names, cmap="coolwarm")
plt.xlabel("Image")
plt.ylabel("Text prompt")
plt.title("CLIP text-image cosine similarity")
plt.show()

# Expected pattern (Part 4.4): "bird" text should score highest against the bird image AND noticeably
# high against the crow image (visually/semantically related); "car" and "bus" should cluster together
# and sit far from "bird"/"crow" -- this clustering IS the learned embedding geometry from Part 1.


---
## Part C — Qwen2.5-VL Image Captioning, Annotated End-to-End

> Also requires a GPU + internet in your bootcamp environment (Qwen2.5-VL-3B is ~3B parameters — noticeably heavier than CLIP's ~82M). This is Part 5 of the theory doc made concrete: fused vision-language Transformer + autoregressive decoding + chat templates.


In [ ]:
# pip install transformers accelerate qwen-vl-utils pillow requests

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info   # official Qwen helper: extracts/formats image inputs
import torch
from PIL import Image
import requests, io

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

# torch_dtype="auto": picks fp16/bf16 on GPU (half the memory of fp32, negligible quality loss for
# inference) and falls back to fp32 on CPU automatically.
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id, torch_dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)


In [ ]:
# ---- Load an image ----
url = "https://example.com/your_image.jpg"   # replace with a real image URL, or use Image.open(local_path)
img_bytes = requests.get(url, timeout=15).content
image = Image.open(io.BytesIO(img_bytes)).convert("RGB")


### C.1 Building the chat-formatted prompt

This is Part 5.4 of the theory doc in code: three roles (system / user / assistant), and images live *inside* the `content` list of a user turn, interleaved with text — this is exactly what "early/mid fusion" (Part 5.2) looks like at the input level, before a single Transformer layer has even run.


In [ ]:
messages = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "You are a helpful assistant that writes concise, accurate image captions."}
        ],
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},                       # the image token(s), Part 2.2 + 5.2
            {"type": "text", "text": "Describe this image in one detailed sentence."},
        ],
    },
]


### C.2 The inference function, explained line by line

Every VLM inference call follows the same three-stage pattern, regardless of framework:

1. **Template the conversation into one string** the model was trained to expect (`apply_chat_template`) — Part 5.4's "chat template" concept.
2. **Separate out and preprocess the visual content** (`process_vision_info`) — resize/patch the image (Part 2.2), independent of the text.
3. **Merge text + vision into one tensor bundle** (`processor(...)`) and **generate** — the autoregressive loop from Part 5.3, producing one token at a time until the model emits an end-of-sequence token.


In [ ]:
def inference(model, processor, msgs, max_new_tokens=128):
    # Stage 1: turn the structured role-based messages into the exact text format Qwen was
    # instruction-tuned on (special tokens marking where system/user/assistant turns begin/end).
    text_prompt = processor.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
        # add_generation_prompt=True appends the tokens that tell the model "now it's your turn to speak"
    )

    # Stage 2: pull the actual image (and/or video) objects out of the message structure and get them
    # into the tensor format the vision encoder expects.
    image_inputs, video_inputs = process_vision_info(msgs)

    # Stage 3: pack text tokens AND image tokens into one aligned set of tensors -- this is the point
    # where the "sequence of tokens" unification from Part 2.2 becomes literal: text and image both
    # end up as entries in the same input_ids-aligned structure.
    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        # Autoregressive generation (Part 5.3): the model predicts one token, appends it, feeds the
        # extended sequence back in, and repeats -- up to max_new_tokens times or until it emits
        # its end-of-sequence token.
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # generated_ids includes the ORIGINAL prompt tokens followed by the new ones -- trim the prompt
    # portion off so we decode only what the model actually generated.
    trimmed_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        trimmed_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]

caption = inference(model, processor, messages)
print("Generated caption:", caption)


---
## Part D — Qwen2.5-VL Object Detection, Annotated End-to-End

This is Part 7 of the theory doc made concrete: **there is no detection architecture here at all.** We reuse the exact `inference()` function from Part C — the only thing that changes is the system prompt, which constrains the *generated text* to be parseable coordinates.


### D.1 The system prompt is the entire "detection head"

Compare this to a classical detector (Faster R-CNN, YOLO), which needs anchor boxes, a region-proposal network, and a non-max-suppression post-processing step baked into the architecture. Here, the "architecture" for detection is a sentence:


In [ ]:
detection_system_prompt = (
    "You are an object detector. The format of your output should be a valid JSON array of objects, "
    "each with the form {'bbox_2d': [x1, y1, x2, y2], 'label': 'class'}, where [x1, y1] is the top-left "
    "corner and [x2, y2] is the bottom-right corner of the bounding box, and 'class' is the object name. "
    "Output ONLY the JSON array, with no additional commentary."
)

detect_messages = [
    {"role": "system", "content": [{"type": "text", "text": detection_system_prompt}]},
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Detect all elephants in this image."},
        ],
    },
]

raw_output = inference(model, processor, detect_messages, max_new_tokens=512)
print(raw_output)


### D.2 Why you need a robust JSON parser (and what "robust" means here)

Per Part 7's limitations section: the model has **no formal guarantee** its output is valid JSON — it is optimized to produce *plausible next tokens*, not syntactically verified structures. In practice it is very good at this (thanks to heavy JSON exposure during instruction tuning) but not perfect. Common failure modes worth defending against:

- Markdown code fences (```json ... ```) wrapped around the array
- Trailing commentary before/after the JSON
- Literal newline characters inside what should be a single-line JSON string (breaks strict JSON parsers)
- Single quotes instead of double quotes (the model may mirror the prompt's `'bbox_2d'` style, which isn't valid JSON)

The parser below defends against exactly these four, each commented at the point it's handled:


In [ ]:
import re
import json as json_lib

def extract_json(raw_text):
    """
    Robustly extract a JSON array from a VLM's raw text output.
    Handles the common failure modes described above, one regex/step at a time.
    """
    text = raw_text.strip()

    # 1. Strip markdown code fences if the model wrapped its answer in ```json ... ```
    fence_match = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if fence_match:
        text = fence_match.group(1).strip()

    # 2. Isolate just the [ ... ] array in case there's leading/trailing commentary
    #    ("Here are the detected objects: [...]") -- take the first '[' to the last ']'.
    start, end = text.find("["), text.rfind("]")
    if start != -1 and end != -1:
        text = text[start:end + 1]

    # 3. Fix single-quoted keys/strings -> valid JSON requires double quotes.
    #    This is a light heuristic, not a full JSON5 parser -- good enough for this model's typical output.
    text = text.replace("'", '"')

    # 4. Attempt to parse; if it fails, surface the cleaned text for debugging rather than crashing silently.
    try:
        return json_lib.loads(text)
    except json_lib.JSONDecodeError as e:
        print("Could not parse model output as JSON. Cleaned text was:\n", text)
        raise e

bounding_boxes = extract_json(raw_output)
print(bounding_boxes)


### D.3 Drawing the result

The final step is pure image-processing housekeeping — worth including because it's where "did the model actually understand geometry" becomes visually checkable, closing the loop back to Part 7's honesty about coordinate imprecision.


In [ ]:
from PIL import ImageDraw, ImageFont

def draw_bboxes(img, boxes, color="red", width=3):
    """
    img: PIL.Image
    boxes: list of {'bbox_2d': [x1, y1, x2, y2], 'label': str}
    """
    img = img.copy()
    draw = ImageDraw.Draw(img)
    for box in boxes:
        x1, y1, x2, y2 = box["bbox_2d"]
        label = box.get("label", "")
        draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
        # small filled label background so text stays legible over busy image regions
        draw.rectangle([x1, y1 - 18, x1 + 8 * len(label), y1], fill=color)
        draw.text((x1 + 2, y1 - 17), label, fill="white")
    return img

output_image = draw_bboxes(image, bounding_boxes)
output_image.show()  # or display(output_image) in a notebook / plt.imshow(output_image)


---
## Where to go from here

- Re-run Part A with different random seeds and dimensionalities — watch how the contrastive loss gap between "aligned" and "random" pairs changes, and connect that back to why CLIP needed *400 million* pairs (Part 4.2) rather than a few hundred: the more negatives per batch, the sharper and more reliable the learning signal.
- In Part B, swap `class_names` for a completely different, unrelated set of words with no retraining, and confirm zero-shot classification still works.
- In Part D, try an object category the model likely never saw labeled in detection-specific training data (Part 7, point 3) and see whether zero-shot generalization holds.
- Read Part 9 of the theory doc *after* running these — the failure modes (hallucination, coordinate imprecision, prompt sensitivity) are far more concrete once you've watched real output come out of `extract_json` needing repair.
